# ACS visualization examples — all 11 pages

Run using your own Census API key. **No synthetic ACS estimates** will be presented as official results. This notebook uses the included public scripts and saves PNG, SVG, CSV, HTML, and provenance files. Upload the full ZIP first.

In [ ]:
from pathlib import Path
import os, zipfile
from getpass import getpass

# Use an already extracted folder locally, or upload the ZIP in Colab.
project = Path.cwd() / "ACS_Examples_Builder"
if not (project / "build_acs_examples.py").exists():
    try:
        from google.colab import files
        print("Upload ACS_Examples_All_11_Python.zip")
        uploaded = files.upload()
        zip_path = next((Path.cwd()/name for name in uploaded if name.endswith(".zip")), None)
        if zip_path is None:
            raise FileNotFoundError("Please upload the ACS examples ZIP file")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(Path.cwd())
    except ImportError:
        raise FileNotFoundError("Place or extract ACS_Examples_Builder alongside this notebook")
assert (project / "build_acs_examples.py").exists(), project
print("Project located:", project)

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(project/"requirements.txt")])
print("Python dependencies available")

In [ ]:
if not os.getenv("CENSUS_API_KEY"):
    os.environ["CENSUS_API_KEY"] = getpass("Enter your own activated Census API key (hidden): ")
if not os.environ["CENSUS_API_KEY"].strip():
    raise ValueError("An activated Census API key is required in API mode")
print("Key configured for this session; it will not be displayed or saved")

In [ ]:
# Run all pages; use ["scatter_plot"] to restrict to a single page.
ONLY_PAGES = None
OUTPUT_ROOT = project / "acs_generated_examples"
cmd = [sys.executable, str(project / "build_acs_examples.py"), "--output-root", str(OUTPUT_ROOT), "--continue-on-error"]
if ONLY_PAGES:
    cmd += ["--only", *ONLY_PAGES]
# Capture and DISPLAY runner messages rather than hiding the errors.
result = subprocess.run(cmd, cwd=project, capture_output=True, text=True)
print("Runner output:\n", result.stdout[-18000:] or "(no output)")
if result.stderr.strip():
    print("Runner diagnostics:\n", result.stderr[-4000:])
print("Process return code:", result.returncode)
if result.returncode != 0:
    print("At least one chart failed. Review the status table and per-chart execution.log files below.")


In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown
status = OUTPUT_ROOT / "execution_status.csv"
if status.exists():
    results = pd.read_csv(status).fillna("")
    display(results[["key", "status", "reason", "log_file"]])
    print("Generated:", (results.status == "generated_unreviewed").sum(),
          "| Failed:", (results.status == "failed").sum())
    pngs = sorted((OUTPUT_ROOT / "assets" / "examples").rglob("*.png"))
    print(f"PNG files generated: {len(pngs)}")
    for image_path in pngs:
        display(Markdown(f"### {image_path.parent.name}: {image_path.name}"))
        display(Image(filename=str(image_path), width=780))
    if not pngs:
        print("No charts were generated. See the reasons above; do not interpret an empty ZIP as success.")
else:
    print("No status report found. See the runner output and diagnostic messages in the previous cell.")


In [ ]:
# Optional: download only successfully produced files (does not modify chart values)
import shutil
if OUTPUT_ROOT.exists():
    target=shutil.make_archive(str(project/"ACS_Actual_Results"),"zip", root_dir=OUTPUT_ROOT)
    print("Results archive:", target)
    try:
        from google.colab import files
        files.download(target)
    except ImportError:
        print("Open this ZIP from your local filesystem")